In [38]:
# Required Libraries
!pip install pandas numpy altair pyspark vega_datasets

# Imports
import pandas as pd
import altair as alt
from vega_datasets import data
import numpy as np

# Heading for the Dashboard
print('📊 Ocean Microbiome Interactive Dashboard')
print('Explore microbial abundance across the worlds oceans with interactive visualizations.')

# Reading the Datasets
meta_deta_df = pd.read_csv('Tara_SampleMeta.csv')
tara_df =pd.read_csv('Tara_OTUtableTax_80CAb.csv')

# Display Initial Rows for Validation
print('Top 5 values of the Meta Data dataset:')
display(meta_deta_df.head())
print('Top 5 values of the Tara OTU table tax - 80CAb dataset:')
display(tara_df.head())

# Data Cleaning
# Removing Unnecessary Columns
tara_df.drop(columns=['Domain','Phylum','Order','Family','Genus','OTU_rep'], inplace=True)

# Calculating Mean Abundance by Class
mean_tara_df = tara_df.groupby('Class').mean().reset_index()

# Transposing the Tara DataFrame
transpose_tara_df = mean_tara_df.T
transpose_tara_df.columns = transpose_tara_df.iloc[0]
transpose_tara_df = transpose_tara_df.drop(transpose_tara_df.index[0])
transpose_tara_df = transpose_tara_df.drop(transpose_tara_df.index[0:6]).reset_index()
transpose_tara_df = transpose_tara_df.rename(columns={'index': 'SampleID'})
display(transpose_tara_df.head())

# Reshaping Data with Melt
melt_tara_df = transpose_tara_df.melt(id_vars='SampleID', var_name='Class', value_name='Abundance')

# Merging Meta Data and Abundance Data
merged_df = pd.merge(meta_deta_df, melt_tara_df, on='SampleID', how='left')
merged_df = merged_df.drop(merged_df[merged_df['Class'] == 'undef'].index)

# Renaming Columns for Consistency
merged_df = merged_df.rename(columns={'SamplingDepth[m]': 'SamplingDepth',
                                       'Latitude[degreesNorth]': 'Latitude',
                                       'Longitude[degreesEast]': 'Longitude'})
merged_df = merged_df.dropna()

# Defining Interactive Selections
class_selection = alt.selection_single(fields=['Class'], name='Select', empty='all')
year_selection = alt.selection_single(fields=['Year'], name='YearSelect', empty='all')

#class_color = alt.condition(class_selection, alt.Color('Class:N', scale=alt.Scale(scheme='tableau20')), alt.value('lightgray'))

#Define Class Selection and Consistent Color Mapping
#class_selection = alt.selection_single(fields=['Class'], name='Select', empty='all')
class_color = alt.condition(
    class_selection,
    alt.Color('Class:N', scale=alt.Scale(scheme='tableau20')),
    alt.value('lightgray')
)


# Interactive Legend for Class
# Interactive Legend for Class (Top Placement with 3 Columns and 10 Rows)
# Updated Legend Configuration for Top Placement
# Create the Legend without Configuration
# Updated Interactive Legend for Class
# Interactive Legend for Class with Consistent Color Mapping
legend = alt.Chart(merged_df).mark_point().encode(
    x=alt.X('Class:N', axis=alt.Axis(orient='top'), sort=None),
    shape=alt.Shape('Class:N', legend=None),
    color=class_color
).properties(
    width=800,
    height=100,
    title='Select Class'
).add_params(class_selection)

# Year Filter Dropdown
year_dropdown = alt.Chart(merged_df).mark_point().encode(
    y=alt.Y('Year:O', axis=alt.Axis(orient='right')),
    shape=alt.Shape('Year:N', legend=None),
    color=alt.condition(year_selection, alt.Color('Year:N', scale=alt.Scale(scheme='category20')), alt.value('lightgray'))
).add_params(year_selection)

# Bar Chart Visualization
bar_chart = alt.Chart(merged_df).mark_bar().encode(
    x='OceanAndSeaRegion:N',
    y='Abundance:Q',
    color=class_color,
    tooltip=['OceanAndSeaRegion', 'Class', 'Abundance']
).properties(
    width=600,
    height=400,
    title='Abundance in Ocean and Sea Regions by Class'
).transform_filter(class_selection)

# Additional Line Chart Visualization
line_chart = alt.Chart(merged_df).mark_line().encode(
    x='Year:O',
    y='average(Abundance):Q',
    color=class_color,
    tooltip=['Year', 'average(Abundance)', 'Class']
).properties(
    width=600,
    height=400,
    title='Abundance Trend Over Years by Class'
).transform_filter(class_selection)

# Scatter Plot Visualization with Class and Year Filtering
scatter_chart = alt.Chart(merged_df).mark_point().encode(
    x='SamplingDepth:Q',
    y='Abundance:Q',
    color=class_color,
    shape='Class:N',
    tooltip=['Abundance', 'SamplingDepth', 'Class', 'LayerOfOrigin', 'Year']
).properties(
    width=600,
    height=400,
    title='Abundance vs Sampling Depth by Class and Year'
).transform_filter(class_selection).transform_filter(year_selection)



# Pie Chart Visualization
pie_chart = alt.Chart(merged_df).mark_arc().encode(
    theta=alt.Theta(field='Abundance', type='quantitative'),
    color=alt.Color(field='Class', type='nominal'),
    tooltip=['Class', 'Abundance']
).properties(
    width=400,
    height=400,
    title='Class Distribution by Abundance'
)

# Geographic Visualization with Map
countries = alt.topo_feature(data.world_110m.url, 'countries')
background = alt.Chart(countries).mark_geoshape(
    fill='lightgray',
    stroke='white'
).properties(
    width=1000,
    height=500
).project('equirectangular')

geo_scatter = alt.Chart(merged_df).mark_point().encode(
    longitude='Longitude:Q',
    latitude='Latitude:Q',
    size=alt.Size('Abundance:Q', legend=None),
    color=class_color,
    tooltip=['OceanAndSeaRegion:N', 'Class', 'Abundance', 'Year'],
    shape='Class:N'
).properties(
    width=1000,
    title='Class Microbes in Ocean and Sea Regions'
).transform_filter(class_selection).transform_filter(year_selection)

# Combining Map with Data Points
map_chart = alt.layer(
    background,
    geo_scatter
)

# Uncertainty Visualization: Scatter Plot with Error Bars
scatter_with_error = alt.Chart(merged_df).mark_point().encode(
    x='SamplingDepth:Q',
    y='Abundance:Q',
    color=class_color,
    tooltip=['SamplingDepth', 'Abundance', 'Class']
).properties(
    width=600,
    height=400,
    title='Abundance vs Sampling Depth with Uncertainty'
)

# Heatmap Visualization
heatmap = alt.Chart(merged_df).mark_rect().encode(
    x='SampleID:O',
    y='Class:O',
    color=alt.Color('Abundance:Q', scale=alt.Scale(scheme='viridis')),
    tooltip=['SampleID', 'Class', 'Abundance']
).properties(
    width=600,
    height=400,
    title='Heatmap of Abundance by Sample and Class'
)

# Save Each Visualization Separately
map_chart.save('map_chart.html')
bar_chart.save('bar_chart.html')
scatter_chart.save('scatter_chart.html')
line_chart.save('line_chart.html')
pie_chart.save('pie_chart.html')
legend.save('legend.html')
year_dropdown.save('year_dropdown.html')

# Combining All Charts into a Dashboard
# Combining All Charts into a Dashboard with Legend and Year Dropdown on Top
dashboard = alt.vconcat(
    alt.hconcat(legend, year_dropdown),  # Move both legend and year dropdown to the top
    map_chart,
    alt.hconcat(bar_chart, scatter_chart),
    alt.hconcat(line_chart, pie_chart),
    scatter_with_error,
    heatmap
).configure_legend(
    columns=10,
    symbolLimit=30,
    orient='top'
)


# Save the Dashboard as an HTML File
dashboard.save('dashboard.html')

print('✅ Dashboard saved as dashboard.html!')
print('✅ Separate visualizations saved as HTML files!')


📊 Ocean Microbiome Interactive Dashboard
Explore microbial abundance across the worlds oceans with interactive visualizations.
Top 5 values of the Meta Data dataset:


,SampleID,Year,Month,Latitude[degreesNorth],Longitude[degreesEast],SamplingDepth[m],LayerOfOrigin,MarinePelagicBiomes,OceanAndSeaRegion,MarinePelagicProvince
0,TARA_004_DCM_0_22_1_6,2009,9,36.5533,-6.5669,40,DCM,Westerlies Biome,North Atlantic Ocean,North Atlantic Subtropical Gyral Province
1,TARA_004_SRF_0_22_1_6,2009,9,36.5533,-6.5669,5,SRF,Westerlies Biome,North Atlantic Ocean,North Atlantic Subtropical Gyral Province
2,TARA_007_DCM_0_22_1_6,2009,9,37.0541,1.9478,42,DCM,Westerlies Biome,Mediterranean Sea,"Mediterranean Sea, Black Sea Province"
3,TARA_007_SRF_0_22_1_6,2009,9,37.0510,1.9378,5,SRF,Westerlies Biome,Mediterranean Sea,"Mediterranean Sea, Black Sea Province"
4,TARA_009_DCM_0_22_1_6,2009,9,39.0609,5.9422,55,DCM,Westerlies Biome,Mediterranean Sea,"Mediterranean Sea, Black Sea Province"


Top 5 values of the Tara OTU table tax - 80CAb dataset:


,Domain,Phylum,Class,Order,Family,Genus,OTU_rep,TARA_018_DCM_0_22_1_6,TARA_018_SRF_0_22_1_6,TARA_023_DCM_0_22_1_6,...,TARA_085_MES_0_22_3,TARA_085_SRF_0_22_3,TARA_093_DCM_0_22_3,TARA_093_SRF_0_22_3,TARA_094_SRF_0_22_3,TARA_096_SRF_0_22_3,TARA_098_DCM_0_22_3,TARA_098_MES_0_22_3,TARA_098_SRF_0_22_3,TARA_099_SRF_0_22_3
0,undef,undef,undef,undef,undef,undef,unclassified,4.400259,3.976295,4.021928,...,4.728833,4.147903,1.208198,0.911905,1.829139,2.997120,6.606668,8.807279,2.899934,2.876148
1,Bacteria,Proteobacteria,Alphaproteobacteria,Rhodospirillales,Rhodospirillaceae,AEGEAN-169 marine group,AACY024102418_157_1623,0.880742,1.008665,0.661216,...,0.030721,0.001225,0.173749,0.191803,2.128909,2.402813,0.724638,0.132403,2.046687,1.890367
2,Bacteria,Cyanobacteria,Cyanobacteria,SubsectionI,FamilyI,Prochlorococcus,KC003383_1_1321,2.326504,1.930531,0.514279,...,0.000000,0.000000,0.013943,0.131234,0.848433,0.722964,2.298567,0.001991,0.594645,1.132555
3,Bacteria,Proteobacteria,Alphaproteobacteria,Rhodospirillales,Rhodospirillaceae,AEGEAN-169 marine group,EU394547_1_1451,0.521889,0.680175,1.917715,...,0.028527,0.000000,1.428602,1.316823,0.896177,0.685683,0.675324,0.230958,0.517203,0.502779
4,Bacteria,Cyanobacteria,Cyanobacteria,SubsectionI,FamilyI,Prochlorococcus,X52169_1_1473,3.265905,2.747639,0.423857,...,0.000000,0.000000,0.008580,0.111044,1.120808,0.858200,1.239692,0.000000,0.103717,0.550663


Class,SampleID,AEGEAN-245,ARKICE-90,Acidimicrobiia,Actinobacteria,Alphaproteobacteria,Arctic97B-4 marine group,Betaproteobacteria,Chloroplast,Cyanobacteria,...,Planctomycetacia,SAR202 clade,Sc-EA05,Skagenf62,Spartobacteria,Subgroup 26,TA18,Thermoplasmata,Verrucomicrobiae,undef
0,TARA_030_DCM_0_22_1_6,0.001267,0.132402,0.058514,0.014254,0.089493,0.007496,0.050522,0.003223,0.083258,...,0.019639,0.012037,0.0,0.003168,0.041811,0.0,0.024073,0.001402,0.0,4.465547
1,TARA_030_SRF_0_22_1_6,0.0,0.040757,0.048474,0.022991,0.089064,0.000174,0.111732,0.002897,0.02149,...,0.016198,0.013324,0.0,0.0,0.06427,0.0,0.014631,0.000311,0.0,3.508204
2,TARA_031_SRF_0_22_1_6,0.000401,0.092318,0.048583,0.003612,0.07262,0.003479,0.037596,0.010816,0.255545,...,0.018464,0.020136,0.0,0.005619,0.020069,0.0,0.08991,0.035134,0.003746,3.057743
3,TARA_032_DCM_0_22_1_6,0.014537,0.199585,0.042507,0.005107,0.048981,0.058147,0.021281,0.015288,0.109038,...,0.008251,0.271614,0.0,0.008643,0.024359,0.0,0.07622,0.134483,0.022263,8.709454
4,TARA_032_SRF_0_22_1_6,0.000577,0.117782,0.093754,0.007506,0.070903,0.004619,0.038683,0.01884,0.209447,...,0.027713,0.017417,0.0,0.003464,0.017321,0.0,0.058891,0.034715,0.00154,4.509186


C:\Users\appuk\AppData\Local\Temp\ipykernel_13264\2696195456.py:53: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use selection_point instead.
  class_selection = alt.selection_single(fields=['Class'], name='Select', empty='all')
C:\Users\appuk\AppData\Local\Temp\ipykernel_13264\2696195456.py:54: AltairDeprecationWarning: 
Deprecated since `altair=5.0.0`. Use selection_point instead.
  year_selection = alt.selection_single(fields=['Year'], name='YearSelect', empty='all')


✅ Dashboard saved as dashboard.html!
✅ Separate visualizations saved as HTML files!
